In [19]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp
from helpers import get_table, get_bronze
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
print(pyspark.__version__)

3.5.8


In [20]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

## Silver license_keys

+ Read Bronze license_keys data
+ Quarantine duplicates for business gain: Deduplicate records to keep one row per license_id, using updated_at and ingest_time to retain the latest version.
+ Cast license_id, subscription_id, max_seats, issued_date, expiry_date, and updated_at to consistent Silver data types.
+ Standardize timestamp fields and derive issued_date_date, issued_date_year, issued_date_month, and expiry_month using a consistent timezone.
+ Normalize technical ingestion metadata by keeping one canonical ingestion timestamp field.
+ Filter out records with null or invalid license_id, subscription_id, issued_date, or expiry_date and quarantine the removing ones.
+ Validate referential integrity by ensuring subscription_id exists in the Silver subscriptions table.
+ Enforce domain rules by quarantine records where max_seats is outside the valid range of 5 to 500.
+ Validate date logic by removing records where expiry_date is earlier than issued_date.
+ Derive business attributes: seat_tier, license_active_flag, days_to_expiry, expiry_month, and seat_utilization_possible_flag.
+ Store the clean result as silver_license_keys_clean in Delta format.
+ Get all quarantined tables
+ Validate post-load row counts and uniqueness of license_id against Bronze input.

## Read Bronze license_keys data

In [21]:
# Bronze layer
license_keys_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/license_keys"
df_bronze_license_keys = get_bronze(license_keys_bronze, spark=spark)
df_bronze_license_keys.show(5)

+----------+---------------+---------+-----------+-----------+--------------------+--------------------+--------------------+
|license_id|subscription_id|max_seats|issued_date|expiry_date|            batch_id|         ingest_time|   source_identifier|
+----------+---------------+---------+-----------+-----------+--------------------+--------------------+--------------------+
|         1|         206618|       41| 2026-09-14| 2027-03-13|app-2026042409000...|2026-04-24 16:00:...|jdbc:postgresql:/...|
|         2|          18978|       11| 2023-08-13| 2024-02-09|app-2026042409000...|2026-04-24 16:00:...|jdbc:postgresql:/...|
|         3|         210644|       42| 2026-01-14| 2027-01-14|app-2026042409000...|2026-04-24 16:00:...|jdbc:postgresql:/...|
|         4|          80642|      170| 2023-10-25| 2024-04-22|app-2026042409000...|2026-04-24 16:00:...|jdbc:postgresql:/...|
|         5|         217968|       94| 2025-01-15| 2025-07-14|app-2026042409000...|2026-04-24 16:00:...|jdbc:postgresq

## Quarantine duplicates for business gain

In [22]:
w = Window.partitionBy("license_id", "issued_date").orderBy(
    F.col("ingest_time").desc()
)

df_ranked = df_bronze_license_keys.withColumn(
    "rn",
    F.row_number().over(w)
)
# clean / filtered
df1 = (
    df_ranked
    .filter(F.col("rn") == 1)
    .drop("rn")
).drop("batch_id", "source_identifier")

# removed duplicates -> quarantine
df1_quarantine = (
    df_ranked
    .filter(F.col("rn") > 1)
    .drop("rn")
)

## Cast types

In [23]:
df2 = df1.select(
    F.col("license_id").cast("bigint").alias("license_id"),
    F.col("subscription_id").cast("bigint").alias("subscription_id"),
    F.col("max_seats").cast("int").alias("max_seats"),
    F.col("issued_date").cast("date").alias("issued_date"),
    F.col("expiry_date").cast("date").alias("expiry_date"),
    F.col("ingest_time").cast("timestamp").alias("ingest_time")
)


In [24]:
# df2.show(2, truncate=False)

## Quarantine NULL data

In [25]:
df_tmp = (
    df2
    .withColumn("issued_date_cast", F.to_date("issued_date"))
    .withColumn("expiry_date_cast", F.to_date("expiry_date"))
)

invalid_condition = (
    F.col("license_id").isNull() |
    F.col("subscription_id").isNull() |
    F.col("issued_date").isNull() |
    F.col("issued_date_cast").isNull() |
    F.col("expiry_date").isNull() |
    F.col("expiry_date_cast").isNull()
)

df3 = (
    df_tmp
    .filter(~invalid_condition)
    .drop("issued_date_cast", "expiry_date_cast")
)

df3_quarantine = (
    df_tmp
    .filter(invalid_condition)
    .drop("issued_date_cast", "expiry_date_cast")
)

In [26]:
# Get the silver_subscriptions
silver_subscriptions_path = "./silver/subscriptions"
df_subscriptions_silver = (
    spark.read
    .format("delta")
    .load(silver_subscriptions_path)
)

df_subscriptions_silver.show(5)

+---------------+-------+-------+----------+----------+---------+-----------+-------------------+
|subscription_id|user_id|plan_id|start_date|  end_date|   status|current_mrr|         created_at|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+
|             14|   6835|      3|2027-04-04|2027-08-18|cancelled|     370.16|2027-04-05 05:21:59|
|             20|    484|      1|2025-05-03|2025-11-30|cancelled|       1.31|2025-05-03 10:30:46|
|             21|  44487|     15|2023-04-11|      NULL|   active|      42.27|2023-04-11 23:55:29|
|             23|   3592|      1|2025-02-10|      NULL|   active|       1.31|2025-02-11 06:43:16|
|             24|  34704|     19|2024-06-20|2024-10-21|  expired|      13.98|2024-06-20 15:07:41|
+---------------+-------+-------+----------+----------+---------+-----------+-------------------+
only showing top 5 rows



## Validate referential integrity with silver subscriptions

In [27]:
valid_subscriptions = df_subscriptions_silver.distinct()
df4 = (
    df3.alias("src")
    .join(
        valid_subscriptions.alias("subs"),
        F.col("src.subscription_id") == F.col("subs.subscription_id"),
        "inner"
    )
    .select("src.*")
)

df4_quarantine = (
    df3.alias("src")
    .join(
        valid_subscriptions.alias("subs"),
        F.col("src.subscription_id") == F.col("subs.subscription_id"),
        "left"
    )
    .filter(F.col("subs.subscription_id").isNull())
    .withColumn("quarantine_reason", F.lit("subscription_id not found in silver_subscriptions"))
    .select("src.*", "quarantine_reason")
)

In [28]:
# df4.count(), df4_quarantine.count()

## Ensure domain rules

In [29]:
df5 = df4.filter(
    col("max_seats").between(5, 500)
)
df5_quarantine = df4.filter(
    ~col("max_seats").between(5, 500)
)
# df5.count(), df5_quarantine.count()

## Validate date logic
By removing records whre expiry_date is earlier than issued_date

In [30]:
df5.show(2)

+----------+---------------+---------+-----------+-----------+--------------------+
|license_id|subscription_id|max_seats|issued_date|expiry_date|         ingest_time|
+----------+---------------+---------+-----------+-----------+--------------------+
|      4538|          71938|       11| 2024-04-02| 2025-04-02|2026-04-24 16:00:...|
|      4537|          71938|       90| 2024-03-25| 2024-09-21|2026-04-24 16:00:...|
+----------+---------------+---------+-----------+-----------+--------------------+
only showing top 2 rows



In [31]:
# Valid records
df6 = df5.filter(
    col("expiry_date") > col("issued_date")
)
# Quarantined records
df6_quarantine = df5.filter(
    col("expiry_date") <= col("issued_date")
)
df6.count(), df6_quarantine.count()

(14422, 0)

In [32]:
silver_license_keys = df6

## Get all quarantine tables

In [33]:
quarantine_dfs = [
    df1_quarantine,
    df3_quarantine,
    df4_quarantine,
    df5_quarantine,
    df6_quarantine
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)
df_quarantine_all.count()

0

## Upsert

Need to think about the case where there are new columns, then, how to upsert?
The current method does not cover that.

In [36]:
silver_path = "./silver/licenses"

# Columns for Silver licenses table
silver_cols = silver_license_keys.columns

# Select only silver columns
df_upsert = silver_license_keys.select(*silver_cols)

# Keep only the latest row per (license_id, updated_at) in the incoming batch
w = (
    Window
    .partitionBy("license_id", "issued_date")
    .orderBy(F.col("ingest_time").desc())
)

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# If Delta table already exists -> MERGE (upsert)
if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (delta_target.alias("target")
    .merge(
        df_upsert.alias("source"),
        """
        target.license_id = source.license_id
        """)
    # Update only if incoming row is newer
    .whenMatchedUpdate(
            condition="source.ingest_time > target.ingest_time",
            set={c: f"source.{c}" for c in silver_cols})
    .whenNotMatchedInsert(values={c: f"source.{c}" for c in silver_cols})
    .execute())
        
# If Delta table does not exist yet -> initial save
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Delta table does not exist yet


In [37]:
# Testing
df_licenses_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_licenses_silver_read.show(5)

+----------+---------------+---------+-----------+-----------+--------------------+
|license_id|subscription_id|max_seats|issued_date|expiry_date|         ingest_time|
+----------+---------------+---------+-----------+-----------+--------------------+
|         1|         206618|       41| 2026-09-14| 2027-03-13|2026-04-24 16:00:...|
|         2|          18978|       11| 2023-08-13| 2024-02-09|2026-04-24 16:00:...|
|         3|         210644|       42| 2026-01-14| 2027-01-14|2026-04-24 16:00:...|
|         4|          80642|      170| 2023-10-25| 2024-04-22|2026-04-24 16:00:...|
|         5|         217968|       94| 2025-01-15| 2025-07-14|2026-04-24 16:00:...|
+----------+---------------+---------+-----------+-----------+--------------------+
only showing top 5 rows



In [38]:
spark.stop()